In [1]:
import re
import numpy as np
import string
import math
import pandas as pd
from nltk.corpus import stopwords
import nltk

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
df = pd.read_csv("spam.csv",encoding="latin1")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mary_\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
def preprocess_text(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '',string.punctuation))
    tokens = text.split()
    text = re.findall(r"[a-z]+", text)
    tokens = [word for word in tokens if word not in stop_words ]
    return tokens

df['tokens'] = df["v2"].apply(preprocess_text)

In [3]:
def build_vocab(token_lists, min_freq = 1):
    vocab = {}
    word_freq = {}
    for tokens in token_lists:
        for token in tokens:
            word_freq[token] = word_freq.get(token, 0) + 1
    idx = 0
    for word, freq in word_freq.items():
        if freq >= min_freq:
            vocab[word] = idx
            idx += 1
    return vocab

vocab = build_vocab(df['tokens'])
print(f"Vocabulary size: {len(vocab)}")

Vocabulary size: 9431


In [4]:
def doc_to_bow(tokens,vocab):
    vec = np.zeros(len(vocab))
    for token in tokens:
        if token in vocab:
            vec[vocab[token]] += 1
    return vec
df['bow_vector']=df['tokens'].apply(lambda x: doc_to_bow(x,vocab))

In [6]:
spam_df = df[df['v1']=='spam']
ham_df = df[df['v1']=='ham']

# posterior probability
P_spam = len(spam_df)/len(df)
P_ham = len(ham_df)/len(df)

spam_tokens_lists = spam_df['tokens'].tolist()
ham_tokens_lists = ham_df['tokens'].tolist()

spam_counts = np.zeros(len(vocab))
ham_counts = np.zeros(len(vocab))

for tokens in spam_tokens_lists:
    for token in tokens:
        if token in vocab:
            spam_counts[vocab[token]] += 1

for tokens in ham_tokens_lists:
    for token in tokens:
        if token in vocab:
            ham_counts[vocab[token]] += 1

spam = spam_counts.sum() #total
ham = ham_counts.sum()
vocab_len = len(vocab)

def p_word_given_class(index, counts, total, V):      # Laplace smoothing
    return (counts[index] + 1) / (total + V)

In [7]:
def classify(tokens):
    log_spam = math.log(P_spam)
    log_ham = math.log(P_ham)

    for token in tokens:
        if token in vocab:
            idx = vocab[token]
            log_spam += math.log(p_word_given_class(idx, spam_counts, spam, vocab_len))
            log_ham += math.log(p_word_given_class(idx, ham_counts, ham, vocab_len))

    return "spam" if log_spam > log_ham else "ham"

In [11]:
test_email1 =  "Congratulations! You've won a free iPhone!"
test_email2 = "Please find attached the report for our meeting."


print("Test Email 1:", classify(preprocess_text(test_email1)))  # probably spam
print("Test Email 2:", classify(preprocess_text(test_email2)))  # probably ham


Test Email 1: spam
Test Email 2: ham
